<a href="https://colab.research.google.com/github/GustavoTriaquim/Estrutura-de-dados-nao-lineares/blob/main/AULA04/Aula04_E01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install osmnx folium -q

import osmnx as ox
import networkx as nx
import folium

In [ ]:
centro_fazenda = [-25.5600, -49.2520]
mapa_fazenda = folium.Map(
    locaton=centro_fazenda,
    zoom_start=16,
    tiles="OpenStreetMap"
)

# Área da fazenda: decágono fictício
area_fazenda = [
    [-25.5580, -49.2540],
    [-25.5575, -49.2520],
    [-25.5585, -49.2500],
    [-25.5600, -49.2490],
    [-25.5615, -49.2495],
    [-25.5625, -49.2505],
    [-25.5628, -49.2525],
    [-25.5620, -49.2545],
    [-25.5605, -49.2555],
    [-25.5590, -49.2555],
]

folium.Polygon(
    locations=area_fazenda,
    color="black",
    weight=2,
    fill=True,
    fill_color="tan",
    fill_opacity=0.3,
    tooltip="Área da fazenda"
).add_to(mapa_fazenda)

# Vértices = registros de irrigação
vertices_irrigacao = [
    [-25.5590, -49.2530],
    [-25.5595, -49.2520],
    [-25.5600, -49.2510],
    [-25.5605, -49.2520],
    [-25.5610, -49.2530],
    [-25.5605, -49.2540],
    [-25.5600, -49.2540],
]

for i, v in enumerate(vertices_irrigacao):
  folium.CircleMarker(
      location=v,
      radius=4,
      color="blue",
      fill=True,
      fill_opacity=0.9,
      popup=f"Registro {i+1}"
  ).add_to(mapa_fazenda)

# Arestas = encanamento, saindo do reservatório central (primeiro vértice)
reservatorio = vertices_irrigacao[0]
for v in vertices_irrigacao[1:]:
  folium.PolyLine(
      locations=[reservatorio, v],
      color="green",
      weight=3,
      opacity=0.8,
      tooltip="Cano de irrigação"
  ).add_to(mapa_fazenda)

mapa_fazenda

In [ ]:
bairro = "Boqueirão, Curitiba, Paraná, Brasil"

G_bairro = ox.graph_from_place(
    bairro,
    network_type="drive",
    truncate_by_edge=True,
    retain_all=True,
    simplify=False
)

mapa_bairro = folium.Map(
    location=ox.geocode(bairro),
    zoom_start=15
)

for indice in G_bairro.nodes:
  posicao = G_bairro.nodes[indice]

  folium.CircleMarker(
      location=[posicao['y'], posicao['x']],
      radius=1,
      color="green",
      fill=True,
      fill_color="green"
  ).add_to(mapa_bairro)

for elemento in G_bairro.edges:
  inicio = [G_bairro.nodes[elemento[0]]['y'], G_bairro.nodes[elemento[0]]['x']]
  fim = [G_bairro.nodes[elemento[1]]['y'], G_bairro.nodes[elemento[1]]['x']]
  folium.PolyLine(
      locations=[inicio, fim],
      color="blue",
      weight=1,
      opacity=0.6
  ).add_to(mapa_bairro)

mapa_bairro

In [ ]:
def geocode_seguro(nome, tentativas_extra=None):
    """Tenta geocodificar o nome; se falhar, tenta variações alternativas."""
    candidatos = [nome] + (tentativas_extra or [])
    for candidato in candidatos:
        try:
            return ox.geocode(candidato)
        except Exception:
            continue
    raise ValueError(f"Não foi possível geocodificar nenhuma variação de: {nome}")

def rota_no_mapa(G, mapa, origem_nome, destino_nome, cor="blue",
                  origem_alt=None, destino_alt=None):
    origem_coord = geocode_seguro(origem_nome, origem_alt)
    destino_coord = geocode_seguro(destino_nome, destino_alt)
    origem_no = ox.distance.nearest_nodes(G, X=origem_coord[1], Y=origem_coord[0])
    destino_no = ox.distance.nearest_nodes(G, X=destino_coord[1], Y=destino_coord[0])
    rota = nx.shortest_path(G, source=origem_no, target=destino_no, weight="length")
    pontos = [[G.nodes[n]['y'], G.nodes[n]['x']] for n in rota]
    folium.PolyLine(locations=pontos, color=cor, weight=4, opacity=0.85,
                     tooltip=f"{origem_nome} → {destino_nome}").add_to(mapa)
    folium.Marker(pontos[0], popup=origem_nome, icon=folium.Icon(color="green")).add_to(mapa)
    folium.Marker(pontos[-1], popup=destino_nome, icon=folium.Icon(color="red")).add_to(mapa)
    return rota

G_regiao = ox.graph_from_place(
    ["São José dos Pinhais, Paraná, Brasil", "Contenda, Paraná, Brasil"],
    network_type="drive", truncate_by_edge=True, retain_all=True, simplify=True
)

mapa_1_3 = folium.Map(
    location=ox.geocode("São José dos Pinhais, Paraná, Brasil"),
    zoom_start=12
)

rota_no_mapa(G_regiao, mapa_1_3,
             "Unisenai, São José dos Pinhais, PR",
             "Terminal Afonso Pena, São José dos Pinhais, PR", "blue")

rota_no_mapa(G_regiao, mapa_1_3,
             "Rua Antônio Singer, 6751, São José dos Pinhais, PR",
             "Avenida Renault, 1300, São José dos Pinhais, PR", "red",
             origem_alt=["Volkswagen, São José dos Pinhais, PR"],
             destino_alt=["Renault do Brasil, São José dos Pinhais, PR"])

rota_no_mapa(G_regiao, mapa_1_3,
             "Aeroporto Afonso Pena, São José dos Pinhais, PR",
             "Rua Canoinhas, 250, São José dos Pinhais, PR", "purple",
             destino_alt=["PIT Borda do Campo, São José dos Pinhais, PR"])

rota_no_mapa(G_regiao, mapa_1_3,
             "PIT Praça da Juventude, São José dos Pinhais, PR",
             "Rua Paulo Setúbal, 5400, Curitiba, PR", "orange")

mapa_1_3